# Harness Quickstart

This notebook demonstrates genai-tk's shared **harness layer** (`genai_tk.agents.harness`), which normalizes LangChain (react | deep | custom) and DeerFlow behind one abstract session interface (`BaseHarness`) and one Pydantic event model.

Covered:
1. Programmatic LangChain agent creation (no YAML)
2. YAML-based LangChain agent creation via `resolve_profile()`
3. Resolving a DeerFlow profile through the same registry
4. Streaming through the unified `create_harness()` API
5. Attaching an MCP server to a profile

See also: [docs/harness.md](../docs/harness.md), [docs/agents.md](../docs/agents.md).

## 1. Programmatic LangChain agent creation

Build an `AgentProfileConfig` directly in Python — no YAML file needed — then wrap it in a `LangChainHarness`.

In [ ]:
from genai_tk.agents.harness.events import EndEvent, ErrorEvent, TokenEvent, ToolCallEvent, ToolResultEvent
from genai_tk.agents.harness.langchain_harness import LangChainHarness
from genai_tk.agents.langchain.config import AgentProfileConfig

profile = AgentProfileConfig(
    name="adhoc-react",
    type="react",
    llm="parrot_local@fake",  # swap for a real llm id, e.g. "gpt_41mini@openai"
    system_prompt="You are a concise assistant.",
)

harness = LangChainHarness(profile)
print(f"harness.name = {harness.name!r}")

In [ ]:
async def run_turn(harness, message: str, thread_id: str = "demo-1") -> None:
    async for event in harness.astream(message, thread_id=thread_id):
        if isinstance(event, TokenEvent):
            print(event.text, end="", flush=True)
        elif isinstance(event, ToolCallEvent):
            print(f"\n[tool_call] {event.tool_name}({event.args})")
        elif isinstance(event, ToolResultEvent):
            print(f"[tool_result] {event.content[:200]}")
        elif isinstance(event, ErrorEvent):
            print(f"\n[error] {event.message}")
        elif isinstance(event, EndEvent):
            print("\n[end]")


await run_turn(harness, "Say hello in one short sentence.")

## 2. YAML-based LangChain agent creation

Profiles usually live in `config/agents/langchain/*.yaml` (or the bundled `config/examples/agents/langchain/` when no project config exists). `resolve_profile()` merges the profile with `defaults:` and returns a fully-resolved `AgentProfileConfig` — this is exactly what `cli agents run <key>` does internally.

In [ ]:
from genai_tk.agents.langchain.config import load_unified_config, resolve_profile

config = load_unified_config()  # searches config/agents/langchain(.yaml) then bundled examples
print("Available profile keys:", list(config.profiles_dict.keys()))

yaml_profile = resolve_profile(config, "research")
print(f"Resolved profile: name={yaml_profile.name!r} type={yaml_profile.type!r} harness={yaml_profile.harness!r}")

## 3. DeerFlow profile through the same registry

DeerFlow profiles live in `config/agents/deerflow.yaml` (or the bundled examples). They carry a `harness: "deerflow"` discriminator field, mirroring `AgentProfileConfig.harness == "langchain"`. `create_harness()` searches **both** config trees by key.

In [ ]:
from genai_tk.agents.harness import create_harness, list_harness_profiles

refs = list_harness_profiles()
for ref in refs:
    print(f"{ref.harness:10s} {ref.key:25s} {ref.name}")

## 4. Streaming through the unified `create_harness()` API

`create_harness(key)` resolves `key` across both config trees and returns the matching adapter (`LangChainHarness` or `DeerFlowHarness`) — the caller does not need to know or care which one it got.

In [ ]:
# Resolves to a LangChain profile (assuming "research" exists in langchain_agents)
lc_harness = create_harness("research", llm_override="parrot_local@fake")
print(type(lc_harness).__name__)

# Resolves to a DeerFlow profile instead, by name — same function, same call shape
# df_harness = create_harness("Research Assistant")
# print(type(df_harness).__name__)

## 5. Attaching an MCP server to a profile

Both LangChain and DeerFlow profiles reference MCP servers by name from `config/mcp_servers.yaml`. Adding one is a one-line YAML change (or, programmatically, appending to `mcp_servers`):

In [ ]:
profile_with_mcp = AgentProfileConfig(
    name="adhoc-with-mcp",
    type="react",
    llm="parrot_local@fake",
    mcp_servers=["tavily-mcp"],  # must be defined in config/mcp_servers.yaml
)
print(profile_with_mcp.mcp_servers)

```yaml
# config/agents/langchain/deep.yaml — equivalent YAML form
langchain_agents:
  research:
    name: "Research"
    type: deep
    llm: gpt_41@openai
    mcp_servers:
      - tavily-mcp
```

Next: see `harness_middleware_demo.ipynb` for cross-harness middleware (anonymization + sensitivity routing).